# **Recorte automatizado de imágenes satelitales para aislar únicamente la vía**

Este script automatiza el proceso de **recorte de imágenes satelitales (GeoTIFF)** utilizando los shapefiles que delimitan la geometría de las vías. El objetivo principal es **eliminar todo el ruido visual** presente en la imagen original (como vegetación, texto, edificaciones o sombreado), y conservar **solo la porción de imagen que contiene la vía**.

### 🎯 Objetivo del proceso:
Extraer únicamente la sección de la imagen que corresponde a la vía reconstruida, lo cual permite:
- Centrar los análisis posteriores exclusivamente sobre la infraestructura vial.
- Evitar que elementos no deseados interfieran en el entrenamiento de modelos de visión computacional o análisis geoespacial.
- Reducir significativamente el tamaño de los datos y el tiempo de procesamiento.

### ✅ Funcionalidades principales:

- 📌 **Recorte de imágenes por shapefile**: cada imagen `.tif` se recorta utilizando las geometrías de su shapefile correspondiente (ya filtrado y reconstruido).
- 🌐 **Reproyección automática**: si la imagen y el shapefile tienen diferentes sistemas de referencia espacial (CRS), el script los alinea automáticamente.
- 💾 **Exportación limpia**: la imagen recortada se guarda como un nuevo archivo `.tif` con metadatos consistentes.
- 📋 **Gestión robusta de errores**: los archivos que no puedan procesarse se registran en un archivo de texto (`error_txt`) para su posterior revisión.

### 🛠️ Variables clave:

- `list_sat`: lista de imágenes satelitales (.tif) a recortar.
- `fol_shp`: carpeta con los shapefiles reconstruidos por vía.
- `sat_attr` y `only_road_folder`: definen la ruta de entrada y salida para las imágenes.
- `error_txt`: archivo donde se guardan los nombres de las imágenes que fallaron durante el proceso.

### 🧠 Aplicaciones típicas:

- Análisis de deterioro, invasión u obstrucción sobre infraestructura vial.
- Generación de datasets curados y enfocados para análisis geoespacial.

Este paso es fundamental para obtener **imágenes precisas y libres de ruido**, centradas únicamente en las vías de interés, mejorando significativamente la calidad de cualquier análisis posterior.



In [ ]:
# Importar librerías necesarias
import os, glob

# Definir parámetros de procesamiento
BB = '22'                 # Bounding box en metros (22 metros en cada dirección del segmento)
troncal = 'Troncal9'      # Identificador de la troncal a procesar
z = '21'                  # Nivel de zoom de las imágenes satelitales

# Construir el nombre del folder de imágenes de entrada (imágenes originales tipo Satellite)
sat_attr = "SourceSatellite_Z{}_SegmentMts100_ExtBB{}".format(z, BB)
fol_sat = "./data/IMAGES/{}/{}".format(sat_attr, troncal)#Ruta con las imagenes datelitales

# (Opcional) Rutas alternativas a diferentes tipos de shapefiles (descomentar si se desea cambiar la fuente de máscaras)
terr_z = '19'  # Nivel de zoom con el que se generó el shapefile (más bajo que el raster final)

# Ruta a los shapefiles de las vías ya reconstruidas y limpias (por ejemplo, con gaps rellenos)
fol_shp = "./data/MASKS_VIAS_FILL_Z{}_BB{}\{}".format(terr_z, BB, troncal)

# Construcción del nombre para la carpeta de salida final con solo la vía recortada desde la imagen
only_road_folder = 'ONLY_ROADS_FILL_terrz{}_Z{}_BB{}'.format(terr_z, z, BB)

# Ruta donde se almacenarán las imágenes recortadas únicamente con la vía
fol_sat_road = "./data/IMAGES/{}/{}".format(only_road_folder, troncal)
os.makedirs(fol_sat_road, exist_ok=True)  # Crear la carpeta si no existe

# Obtener lista de imágenes satelitales (.tif) y shapefiles (.shp)
list_sat = sorted(glob.glob(os.path.join(fol_sat, '*.tif')))
list_shp = sorted(glob.glob(os.path.join(fol_shp, '*.shp')))

# Tomar un shapefile y una imagen como ejemplo para procesar
shp = list_shp[0]
sat = list_sat[0]

# Definir ruta de salida para la imagen recortada usando la estructura de carpetas personalizada
via = sat.replace('SourceSatellite_Z21_SegmentMts100_ExtBB{}'.format(BB), only_road_folder)

# Ruta al archivo de texto donde se registrarán imágenes que no puedan ser procesadas
error_txt = fol_sat_road + '_OR_Faltantes.txt'

# Imprimir cantidad de imágenes y shapefiles disponibles
print(len(list_sat))  # Cantidad de imágenes satelitales encontradas
print(len(list_shp))  # Cantidad de shapefiles encontrados


In [ ]:
import os
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from tqdm import tqdm

# Crear un archivo de texto para registrar las imágenes que no se procesan
with open(error_txt, "w") as error_file:
    for sat in tqdm(list_sat):
        shp=os.path.join(fol_shp,os.path.basename(sat).replace('.tif','.shp'))
        try:
            vias_gdf = gpd.read_file(shp)

            # Asegurarse de que el shapefile y la imagen tengan el mismo CRS
            with rasterio.open(sat) as src:
                if str(vias_gdf.crs) != str(src.crs):
                    vias_gdf = vias_gdf.to_crs(src.crs)

            # Convertir las geometrías del shapefile a un formato compatible con rasterio
            geometries = [geom.__geo_interface__ for geom in vias_gdf.geometry]

            # Generar el nombre del archivo de salida
            via = sat.replace(sat_attr, only_road_folder)

            # Verificar si el archivo de salida ya existe
            if not os.path.exists(via):
                # Mostrar mensaje solo cuando el archivo se va a crear
                print(f"Creando archivo: {via}")

                # Abrir la imagen raster y recortar con las geometrías del shapefile
                with rasterio.open(sat) as src:
                    # Recortar la imagen con las geometrías
                    imagen_recortada, transform = mask(src, geometries, crop=True)
                    # Actualizar los metadatos
                    meta = src.meta.copy()
                    meta.update({
                        "driver": "GTiff",
                        "height": imagen_recortada.shape[1],
                        "width": imagen_recortada.shape[2],
                        "transform": transform
                    })

                # Guardar la imagen recortada
                with rasterio.open(via, "w", **meta) as dest:
                    dest.write(imagen_recortada)

        except Exception as e:
            # Si ocurre un error, registrar el archivo con el error en el archivo de texto
            error_file.write(f"Error procesando {sat}: {str(e)}\n")
            print(f"Error procesando {sat}. Se registró en el archivo de errores.")


In [ ]:
#Visualización opcional
import leafmap
import rasterio

# Asegúrate de que se abre correctamente
with rasterio.open(via) as src:
    print("Archivo raster recortado cargado correctamente")
    print(f"Dimensiones: {src.width}x{src.height}")
    print(f"CRS: {src.crs}")

# Crear un mapa con Leafmap
m = leafmap.Map()

# Añadir el raster recortado al mapa
try:
    m.add_raster(via, layer_name="Vía Recortada")
    print("Raster agregado al mapa exitosamente")
except Exception as e:
    print(f"Error al agregar el raster al mapa: {e}")

# Mostrar el mapa
m
